# A0 — The SH3 cancer-variant atlas, rebuilt from raw COSMIC (Stage 0)

**Question.** Which somatic missense mutations in human cancers fall inside an SH3 domain, and
can each one be placed on the correct residue?

**What it does.** Reads the COSMIC Cancer Mutation Census and keeps a missense row only if its
position lies inside a UniProt SH3 domain **and** the wild-type residue COSMIC reports is the
residue actually present there. Rows numbered on an alternative transcript are recovered through
UniProt isoforms. The per-sample genome screens then add tumour type and hypermutation status.

**Inputs** (`input/cosmic/`, `input/uniprot/`) — all MD5-checked before use:
`cmc_export.tsv`, `Cosmic_GenomeScreensMutant_v104_GRCh38.tsv.gz`,
`Cosmic_Classification_v104_GRCh38.tsv`, `sh3_domains_human.tsv`, `sh3_proteins_seq.tsv`,
`isoforms_all.json`. The UniProt files are a frozen 2026-09-20 snapshot, so a later UniProt
release cannot change the result.

**Outputs** (`results/A0_cancer_atlas/`): `SH3_cancer_master.tsv` (the atlas),
`SH3_cancer_unmapped.tsv` (rejected rows), `qc_checks.csv`, `RESULTS_A0.md`.

**Runtime:** 10–25 min on a standard CPU runtime (no GPU needed; 6 min on a laptop). The COSMIC
files are read by streaming; the 16.8 GB genome-screens file is never unpacked.

Explanation of every step: `docs/A0_cancer_atlas.md`.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'A0', 'Cancer-variant atlas from raw COSMIC', resume=RESUME)
print(C.env_info())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A0 — Cancer-variant atlas from raw COSMIC
  results -> /content/drive/MyDrive/sh3_repro/results/A0_cancer_atlas
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'torch': '2.11.0+cpu', 'gpu': 'none'}


## 1. Check the inputs
A wrong or truncated file stops the run here instead of silently changing the atlas (the original project once used a copy of the genome-screens file that had stopped at 1.35 of 16.8 GB).

In [2]:
COSMIC = ['input/cosmic/cmc_export.tsv',
          'input/cosmic/Cosmic_GenomeScreensMutant_v104_GRCh38.tsv.gz',
          'input/cosmic/Cosmic_Classification_v104_GRCh38.tsv']
UNIPROT = ['input/uniprot/sh3_domains_human.tsv', 'input/uniprot/sh3_proteins_seq.tsv',
           'input/uniprot/isoforms_all.json']
R.verify_inputs(UNIPROT + COSMIC)
# optional: the original Stage 0 atlas, used ONLY to prove the rebuild is identical (section 5)
ORIGINAL = f'{R.REF}/SH3_cancer_master_v3.tsv'
HAVE_ORIGINAL = os.path.exists(ORIGINAL)
print('original atlas for comparison:', 'found' if HAVE_ORIGINAL else 'not uploaded - comparison will be skipped')

  verified input/uniprot/sh3_domains_human.tsv  (0s)
  verified input/uniprot/sh3_proteins_seq.tsv  (0s)
  verified input/uniprot/isoforms_all.json  (0s)
  verified input/cosmic/cmc_export.tsv  (27s)
  verified input/cosmic/Cosmic_GenomeScreensMutant_v104_GRCh38.tsv.gz  (58s)
  verified input/cosmic/Cosmic_Classification_v104_GRCh38.tsv  (0s)
original atlas for comparison: not uploaded - comparison will be skipped


## 2. Unit tests of the shared code
Fast checks of mutation parsing, the sign convention, the partition modes and the conformal quantile (`code/run_tests.py`).

In [3]:
rc = os.system(f'cd {ROOT}/code && python run_tests.py')
assert rc == 0, 'unit tests failed - see the output above'

## 3. Build the atlas
The same selection and aggregation rules as the original build (`code/sh3_atlas.py`, standard library only).

In [4]:
import sh3_atlas as S; importlib.reload(S)
if R.done('SH3_cancer_master.tsv', 'build_stats.json'):
    ST = json.load(open(R.out('build_stats.json'))); print('atlas already built in this folder - loaded')
else:
    t = time.time()
    ST = S.build_atlas(f'{ROOT}/{COSMIC[0]}', f'{ROOT}/{COSMIC[1]}', f'{ROOT}/{COSMIC[2]}',
                       *[f'{ROOT}/{p}' for p in UNIPROT], R.OUT)
    ST['minutes'] = round((time.time() - t) / 60, 1)
    json.dump(ST, open(R.out('build_stats.json'), 'w'), indent=1)
print(json.dumps(ST, indent=1))

{
 "windows": {
  "canonical": 325,
  "isoform": 518
 },
 "cmc": {
  "rows": 5802907,
  "missense_sh3_gene": 91842,
  "in_window_residue_mismatch": 490,
  "mapped_canonical": 5923,
  "mapped_isoform": 1034
 },
 "tumour": {
  "gsm_rows": 62765128,
  "gsm_short_rows": 0,
  "gsm_samples": 53598,
  "atlas_hits": 16335,
  "burden_median": 172,
  "burden_p95": 6075,
  "burden_p99": 16293,
  "hypermutated_samples": 9725,
  "hypermutated_fraction": 0.1814433374379641
 },
 "variants": 6884,
 "columns": 44,
 "minutes": 12.6
}


## 4. Data-quality report
Every row is re-checked against its own sequence. A failed check is a bug, not a filter, so the notebook stops if any row fails.

In [5]:
M = pd.read_csv(R.out('SH3_cancer_master.tsv'), sep='\t', keep_default_na=False, dtype=str)
U = pd.read_csv(R.out('SH3_cancer_unmapped.tsv'), sep='\t', keep_default_na=False, dtype=str)
I = lambda c: M[c].astype(int)
checks = {
  'variant_id unique':                         M.variant_id.is_unique,
  'wild-type residue matches domain sequence': all(s[p-1] == w for s, p, w in zip(M.wt_domain_seq, I('pos_in_domain'), M.wt_aa)),
  'mutant sequence differs at one residue':    all(sum(a != b for a, b in zip(w, m)) == 1 and len(w) == len(m)
                                                   for w, m in zip(M.wt_domain_seq, M.mut_domain_seq)),
  'position inside the domain':                bool(((I('pos_in_domain') >= 1) & (I('pos_in_domain') <= I('dom_len'))).all()),
  'protein position = domain start + offset':  bool((I('pos_protein') == I('dom_start') + I('pos_in_domain') - 1).all()),
  'domain length = sequence length':           bool((I('dom_len') == M.wt_domain_seq.str.len()).all()),
  'wild-type differs from mutant':             bool((M.wt_aa != M.mut_aa).all()),
  'standard amino acids only':                 bool(M.wt_aa.isin(list('ACDEFGHIKLMNPQRSTVWY')).all()
                                                    and M.mut_aa.isin(list('ACDEFGHIKLMNPQRSTVWY')).all()),
}
QC = pd.DataFrame({'check': list(checks), 'passed': list(checks.values())})
QC.to_csv(R.out('qc_checks.csv'), index=False); print(QC.to_string(index=False))
assert QC.passed.all(), 'a data-quality check failed'

route = M.position_source.map(lambda s: 'canonical' if 'canonical' in s else 'isoform only')
nh = I('n_samples_nonhyper'); gsm = I('n_samples_gsm')
R.section('Data-quality report', 'Every retained row re-verified against its own sequence.', QC)
R.section('How the variants were mapped', '',
          pd.DataFrame({'route': ['canonical sequence', 'recovered via an isoform', 'rejected: residue mismatch on every sequence'],
                        'COSMIC rows': [ST['cmc'].get('mapped_canonical', 0), ST['cmc'].get('mapped_isoform', 0),
                                        ST['cmc'].get('in_window_residue_mismatch', 0)]}))
R.section('Rejected rows, by gene (top 10)', 'Full list: `SH3_cancer_unmapped.tsv`.',
          U.gene.value_counts().head(10).rename_axis('gene').reset_index(name='rejected rows'))

                                    check  passed
                        variant_id unique    True
wild-type residue matches domain sequence    True
   mutant sequence differs at one residue    True
               position inside the domain    True
 protein position = domain start + offset    True
          domain length = sequence length    True
            wild-type differs from mutant    True
                standard amino acids only    True


## 5. Is it the same atlas as the original?
The rebuilt file's MD5 is compared with the recorded MD5 of the Stage 0 atlas of 2026-09-20. Only if they differ, and the original file was uploaded to `reference/`, are the differing variants and columns listed.

In [6]:
new_md5 = C.md5(R.out('SH3_cancer_master.tsv'))
# the recorded MD5 of the original atlas also sits in expected_values.json, so the check in section 6
# works even without the file; the file itself only adds a row-by-row diff if they ever differ
identical = new_md5 == 'cef897f4dd25f9cd56f2c40bb7ec0b08'
print('byte-identical to the original Stage 0 atlas (by recorded MD5):', identical)
if not identical and HAVE_ORIGINAL:
    O = pd.read_csv(ORIGINAL, sep='\t', keep_default_na=False, dtype=str)
    only_new, only_old = set(M.variant_id) - set(O.variant_id), set(O.variant_id) - set(M.variant_id)
    j = M.merge(O, on='variant_id', suffixes=('', '_old'))
    diff = {c: int((j[c] != j[c + '_old']).sum()) for c in M.columns if c != 'variant_id' and c + '_old' in j}
    print(f'only in new: {len(only_new)} | only in old: {len(only_old)}')
    print('columns that differ:', {k: v for k, v in diff.items() if v})

byte-identical to the original Stage 0 atlas (by recorded MD5): True


## 6. Results

In [7]:
tier = M.mutation_significance_tier
for k, v in [('cmc_rows', ST['cmc']['rows']), ('missense_rows_in_sh3_genes', ST['cmc']['missense_sh3_gene']),
             ('mapped_canonical', ST['cmc'].get('mapped_canonical', 0)), ('mapped_isoform', ST['cmc'].get('mapped_isoform', 0)),
             ('unmapped', len(U)), ('variants', len(M)), ('columns', M.shape[1]),
             ('domains', M.groupby(['uniprot', 'domain_idx']).ngroups), ('genes', M.gene.nunique()),
             ('gsm_rows', ST['tumour']['gsm_rows']), ('gsm_short_rows', ST['tumour']['gsm_short_rows']),
             ('gsm_samples', ST['tumour']['gsm_samples']),
             ('hypermutated_fraction', round(ST['tumour']['hypermutated_fraction'], 4)),
             ('with_sample_records', int((gsm > 0).sum())), ('nonhyper_variants', int((nh > 0).sum())),
             ('recurrent_nonhyper', int((nh >= 3).sum())), ('hypermutated_only', int(((gsm > 0) & (nh == 0)).sum())),
             ('recurrent_any', int((M.is_recurrent == '1').sum())), ('likely_germline', int((M.likely_germline == '1').sum())),
             ('significance_tier_1_3', int(tier.isin(['1', '2', '3']).sum())),
             ('dnds_flagged', int((M.dnds_disease_qval_sig != '').sum())), ('master_md5', new_md5)]:
    R.metric(k, v)
R.section('Hypermutation', f"A sample with more than {S.HYPERMUTATION_THRESHOLD:,} mutations in the genome screens is "
          f"called hypermutated: {ST['tumour']['hypermutated_samples']:,} of {ST['tumour']['gsm_samples']:,} samples "
          f"({ST['tumour']['hypermutated_fraction']:.1%}); median burden {ST['tumour']['burden_median']}.")
R.finish([f"{len(M):,} residue-verified missense variants in {R.metrics['domains']} SH3 domains of {M.gene.nunique()} genes.",
          f"{R.metrics['mapped_isoform']:,} COSMIC rows recovered through alternative isoforms; {len(U)} rejected.",
          f"{R.metrics['nonhyper_variants']:,} variants are seen outside hypermutated tumours; only "
          f"{R.metrics['recurrent_nonhyper']} of them in 3 or more such samples.",
          f"No variant carries a COSMIC significance tier 1-3 or a dN/dS flag - recurrence-based driver "
          f"detection cannot see this domain family.",
          f"Byte-identical to the original atlas: {identical}."])


A0: REPRODUCED
                  quantity                         this_run                         recorded    status
                  cmc_rows                          5802907                          5802907 identical
missense_rows_in_sh3_genes                            91842                            91842 identical
          mapped_canonical                             5923                             5923 identical
            mapped_isoform                             1034                             1034 identical
                  unmapped                              490                              490 identical
                  variants                             6884                             6884 identical
                   columns                               44                               44 identical
                   domains                              310                              310 identical
                     genes                              2

,quantity,this_run,recorded,status
0,cmc_rows,5802907,5802907,identical
1,missense_rows_in_sh3_genes,91842,91842,identical
2,mapped_canonical,5923,5923,identical
3,mapped_isoform,1034,1034,identical
4,unmapped,490,490,identical
5,variants,6884,6884,identical
6,columns,44,44,identical
7,domains,310,310,identical
8,genes,235,235,identical
9,gsm_rows,62765128,62765128,identical
